# Day 55 · Context engineering: what the model sees is all it knows

**First-principles question:** What should go into the context, and what should stay out?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import importlib
import numpy as np
import pandas as pd
import lab.agent as agent
from lab.acme import load_cases
from lab.evals import score

cases = load_cases()

## 1 · Where do the tokens go?

In [ ]:
per_step = []
for case in cases:
    answer, trace = agent.run_agent(case)
    for e in trace:
        per_step.append({"case": case["id"], "step": e["step"], "tokens_in": e["tokens_in"],
                         "tools": ", ".join(name for name, _, _ in e["tool_calls"])})
steps = pd.DataFrame(per_step)
print(steps.groupby("step").tokens_in.mean().round(0))
print("system prompt alone:", len(agent.SYSTEM_C.split()), "words")

## 2 · Change the context
Ideas: a shorter system prompt; `lookup_customer` returns only the fields the reply needs; `search_kb` returns 1 article instead of 2; drop old tool results after they are used.

In [ ]:
# TODO: make your change in lab/agent.py or lab/acme_tools.py, then describe it here in one line
CHANGE = ""

## 3 · Before and after

In [ ]:
def measure(run):
    importlib.reload(agent)
    rows = []
    for case in cases:
        answer, trace = agent.run_agent(case, run=run)
        rows.append({"passed": score(case, answer)["passed"], "tokens": sum(e["tokens_in"] + e["tokens_out"] for e in trace),
                     "seconds": sum(e["seconds"] for e in trace)})
    df = pd.DataFrame(rows)
    return {"pass rate": df.passed.mean(), "tokens per email": df.tokens.mean(), "seconds per email": df.seconds.mean()}

from lab.acme import load_results
by_id = {c["id"]: c for c in cases}
day9 = load_results("C")       # saved on day 9, before your change
before = {"pass rate": np.mean([score(by_id[r["id"]], r)["passed"] for r in day9]),
          "tokens per email": np.mean([r["tokens_in"] + r["tokens_out"] for r in day9]),
          "seconds per email": np.mean([r["seconds"] for r in day9])}
after = measure(run=2)         # run=2: fresh answers with your change
pd.DataFrame({"before (day 9)": before, "after": after})

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert len(steps) > 0
assert CHANGE.strip(), "Step 2: describe your context change"
assert after["tokens per email"] > 0
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")